# Merge PSW2024 (geopoliticaldistance.org) Country-Pair Lookup

Reads the raw Pellegrino, Spolaore & Wacziarg (2024) linguistic-distance dataset
(`data/gravity/linguistic_distance_PSW2024.csv`) and resolves it into a complete,
closed country-pair lookup over every ISO3 code that appears anywhere in our match
data. Two robustness measures for linguistic proximity, both independent
constructions from `prox1` (Melitz & Toubal 2014, see `merge_linguistic_proximity.ipynb`):

- `psw_tree` -- inverted tree-distance index (`1 - lingdist_tree_weighted`).
- `psw_cognet` -- cognate/lexical-based proximity (`lingprox_CogNet_weighted`).

Unlike `prox1`, PSW2024 has **complete (100%) coverage** for every pair in our
65-country universe directly from the raw file -- no Monaco/KOR fallback logic is
needed here (confirmed directly, 2026-09-11: 0 missing pairs of 2,080).

**Output**: `data/gravity/psw2024_pairs_final.csv` -- a complete lookup requiring no
further logic downstream. Used the same way as `ling_prox_pairs_final.csv`, for any
consumer needing PSW2024 values for arbitrary (not just realized-match) country pairs;
`final_ds.ipynb` reads it too, for the per-match `winners/losers_ling_prox_psw_tree`/
`_psw_cognet` columns baked into `men_matches_with_ranks_cleaned.xlsx`.

**Pipeline position**: independent of `final_ds.ipynb`/`homophily.ipynb` -- run
whenever `linguistic_distance_PSW2024.csv` changes or the country universe expands.

In [1]:
import os
import pandas as pd
from itertools import combinations

ROOT       = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
EXCEL_PATH = os.path.join(ROOT, 'data', 'atp', 'men_matches_with_ranks_cleaned.xlsx')
PSW_PATH   = os.path.join(ROOT, 'data', 'gravity', 'linguistic_distance_PSW2024.csv')
OUT_PATH   = os.path.join(ROOT, 'data', 'gravity', 'psw2024_pairs_final.csv')
print('Paths set.')

Paths set.


## 1. Country universe: every ISO3 appearing in our match data

In [2]:
iso_cols = ['winners_p1_iso3', 'winners_p2_iso3', 'losers_p1_iso3', 'losers_p2_iso3']
raw = pd.read_excel(EXCEL_PATH, sheet_name='players_list', usecols=iso_cols)

isos = set()
for c in iso_cols:
    isos |= set(raw[c].dropna().unique())
isos = sorted(isos)
print(f'Country universe: {len(isos)} ISO3 codes')

Country universe: 65 ISO3 codes


## 2. Raw PSW2024 lookup

In [3]:
psw_raw = pd.read_csv(PSW_PATH)
tree_lut, cognet_lut = {}, {}
for _, r in psw_raw.iterrows():
    if r['countrycode_1'] == r['countrycode_2']:
        continue
    key = tuple(sorted([r['countrycode_1'], r['countrycode_2']]))
    tree_lut[key]   = 1.0 - float(r['lingdist_tree_weighted'])
    cognet_lut[key] = float(r['lingprox_CogNet_weighted'])
print(f'Raw PSW2024 country-pairs: {len(tree_lut):,}')

Raw PSW2024 country-pairs: 29,161


## 3. Build the complete, closed lookup table

In [4]:
rows = []
n_missing = 0
for a, b in combinations(isos, 2):
    key = tuple(sorted([a, b]))
    if key not in tree_lut:
        n_missing += 1
    rows.append({
        'iso3_a': a, 'iso3_b': b,
        'psw_tree':   tree_lut.get(key, float('nan')),
        'psw_cognet': cognet_lut.get(key, float('nan')),
    })
for a in isos:
    rows.append({'iso3_a': a, 'iso3_b': a, 'psw_tree': 1.0, 'psw_cognet': 1.0})

pairs_final = pd.DataFrame(rows).sort_values(['iso3_a', 'iso3_b']).reset_index(drop=True)
print(f'Pairs exported: {len(pairs_final):,} ({len(isos)} countries, self-pairs included)')
print(f'Pairs missing from raw PSW2024 file: {n_missing}')
print(f'psw_tree range:   [{pairs_final["psw_tree"].min():.3f}, {pairs_final["psw_tree"].max():.3f}]')
print(f'psw_cognet range: [{pairs_final["psw_cognet"].min():.3f}, {pairs_final["psw_cognet"].max():.3f}]')
assert pairs_final['psw_tree'].between(0, 1).all() and pairs_final['psw_cognet'].between(0, 1).all(), \
    'psw_tree/psw_cognet must be within [0,1]'
assert pairs_final[['psw_tree', 'psw_cognet']].notna().all().all(), \
    'psw_tree/psw_cognet must have no missing values for this country universe'

Pairs exported: 2,145 (65 countries, self-pairs included)
Pairs missing from raw PSW2024 file: 0
psw_tree range:   [0.000, 1.000]
psw_cognet range: [0.001, 1.000]


## 4. Save

In [5]:
pairs_final.to_csv(OUT_PATH, index=False)
print(f'Saved {len(pairs_final)} rows -> {OUT_PATH}')

Saved 2145 rows -> C:\Users\aldi\Documents\GitHub\tennis-homophily\data\gravity\psw2024_pairs_final.csv
